In [12]:
import json
import librosa
import numpy as np
from glob import glob
from sentence_transformers import SentenceTransformer
from scipy.stats import wilcoxon

from src.models import load_whisper_model

In [2]:
methods = ["TTS", "Waveform"]
paths = ["../outputs/results/" + m for m in methods]
results = [dict() for _ in methods]

## Test Cases Found

In [3]:
jsons = [glob(p + "/**/**/**/*.json") for p in paths]

In [4]:
for m, j in zip(methods, jsons):
    success = 0
    for file in j:
        with open(file, "r") as f:
            d = json.load(f)
        success += bool(d["success_metrics"]["success"])
    print(f"{m}: {success/len(j)*100:.2f}%")

TTS: 69.31%
Waveform: 34.00%


## UTMOS

In [5]:
utmos_gt = []
for m, j, d in zip(methods, jsons, results):
    utmos = []
    for file in j:
        with open(file, "r") as f:
            d = json.load(f)
        utmos.append(d["naturalness_scores"]["utmos_best"])
        utmos_gt.append(d["naturalness_scores"]["utmos_gt"])
    print(f"{m}: {np.mean(utmos):.3f} pm {np.std(utmos):.3f}")
    d |= {"utmos": utmos}
print(f"GT: {np.mean(utmos_gt):.3f} pm {np.std(utmos_gt):.3f}")

TTS: 4.466 pm 0.030
Waveform: 2.108 pm 0.108
GT: 4.473 pm 0.046


## Spectogram distance

In [6]:
gt, best = [glob(p + "/**/**/**/ground_truth.wav") for p in paths], [glob(p + "/**/**/**/best_mixed.wav") for p in paths]

In [7]:
def to_spectogram(file: str):
    y, sr = librosa.load(file, sr=None)

    S_mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=256, n_mels=128)
    S_mel_db = librosa.power_to_db(np.abs(S_mel), ref=S_mel.max())
    return S_mel_db

In [8]:
for m, g, b, d in zip(methods, gt, best, results):
    dists = []
    for g_file, b_file in zip(g, b):
        gspec, bspec = to_spectogram(g_file), to_spectogram(b_file)
        diff = gspec - bspec
        dists.append(np.linalg.norm(diff, ord="fro")/gspec.size)  # Normalized Frob-Dist to size of Spectograms
    d |= {"spec_distances": dists}
    print(f"{m}: {np.mean(dists):.3f} pm {np.std(dists):.3f}")

TTS: 0.017 pm 0.002
Waveform: 0.106 pm 0.023


## Sentence level Distances

In [9]:
whisper = load_whisper_model()
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
def char_cosine(s1, s2):
    v1 = np.array([s1.count(c) for c in set(s1 + s2)])
    v2 = np.array([s2.count(c) for c in set(s1 + s2)])
    if (n1:=np.linalg.norm(v1)) == 0 or (n2:=np.linalg.norm(v2)) == 0:
        return 0.0
    return np.dot(v1, v2) / (n1 * n2)

In [11]:
for m, j, d in zip(methods, jsons, results):
    cos_dist, emb_dist = [], []
    for file in j:
        with open(file, "r") as f:
            jd = json.load(f)
        gtranscr, btranscr = jd["text_data"]["gt_transcription"].lower().strip(), jd["text_data"]["asr_transcription"].lower().strip()
        cos_dist.append(1-char_cosine(gtranscr, btranscr))
        e1, e2 = emb_model.encode(gtranscr), emb_model.encode(btranscr)
        emb_dist.append(1 - np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2)))
    d |= {"cos-dist": cos_dist, "emb-dist": emb_dist}
    print(f"{m} - Cos-Dist: {np.mean(cos_dist):.3f} pm {np.std(cos_dist):.3f}")
    print(f"{m} - Emb-Dist: {np.mean(emb_dist):.3f} pm {np.std(emb_dist):.3f}")

TTS - Cos-Dist: 0.026 pm 0.021
TTS - Emb-Dist: 0.283 pm 0.218
Waveform - Cos-Dist: 0.884 pm 0.298
Waveform - Emb-Dist: 0.870 pm 0.172


## Statistical Analysis

In [16]:
alternative = {
    "utmos": "greater",
    "spec_distances": "less",
    "cos-dist": "less",
    "emb-dist": "less",
}

In [26]:
def cohens_d_paired(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    n = min(len(a), len(b))
    diff = a[:n] - b[:n]
    return abs(np.mean(diff) / np.std(diff, ddof=1))

In [27]:
for i, m in enumerate(results[1:], 1):
    print(f"TTS vs {methods[i]}")
    for k, vals in results[0].items():
        stat, p = wilcoxon(vals[:100], m[k][:100], alternative=alternative[k])
        d = cohens_d_paired(vals[:100], m[k][:100])
        print(f"\t{k}: p={p:.3e}; Cohens-D: {d:.1f}")

TTS vs Waveform
	spec_distances: p=1.948e-18; Cohens-D: 3.8
	cos-dist: p=2.262e-18; Cohens-D: 2.9
	emb-dist: p=1.482e-17; Cohens-D: 2.0
